In [1]:
import pandas as pd
df=pd.read_excel('C:/Develop/深圳42/data/PreferenceAnalysis.xlsx')
df.head()

,用户ID,付款日期,订单状态,实付金额,邮费,省份,城市,购买数量
0,uid00324123,2023-04-18,交易成功,22.32,0,北京,北京市,1
1,uid00324124,2023-02-17,交易成功,87.00,0,上海,上海市,1
2,uid00324125,2023-04-18,交易成功,97.66,0,福建省,福州市,2
3,uid00324126,2023-01-11,交易成功,37.23,0,河南省,安阳市,3
4,uid00324126,2023-02-18,交易成功,29.50,0,河南省,安阳市,2


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28832 entries, 0 to 28831
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   用户ID    28832 non-null  object        
 1   付款日期    28832 non-null  datetime64[ns]
 2   订单状态    28832 non-null  object        
 3   实付金额    28832 non-null  float64       
 4   邮费      28832 non-null  int64         
 5   省份      28832 non-null  object        
 6   城市      28832 non-null  object        
 7   购买数量    28832 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 1.8+ MB


In [3]:
df.describe()

,实付金额,邮费,购买数量
count,28832.000000,28832.000000,28832.000000
mean,54.276731,0.245248,1.510960
std,50.795499,1.386458,1.483501
min,0.005000,0.000000,1.000000
25%,23.748750,0.000000,1.000000
50%,34.500000,0.000000,1.000000
75%,84.500000,0.000000,2.000000
max,4225.000000,18.000000,100.000000


In [5]:
df['用户ID'].value_counts()

uid00325812    16
uid00338903     7
uid00328770     7
uid00345755     5
uid00336495     5
               ..
uid00333199     1
uid00333198     1
uid00333197     1
uid00333195     1
uid00350070     1
Name: 用户ID, Length: 25948, dtype: int64

In [7]:
# 计算每个人的平均消费金额
users_df = df.groupby('用户ID', as_index=False)['实付金额'].mean()
users_df.columns = ['用户ID','平均支付金额']

In [10]:
# 根据平均消费金额给用户贴上标签(高客单价, 低客单价)
users_df['客单价类别'] = users_df['平均支付金额'].apply(lambda x:'高客单价' if x>50 else '低客单价')
users_df['客单价类别'].value_counts()

低客单价    15166
高客单价    10782
Name: 客单价类别, dtype: int64

In [11]:
users_df

,用户ID,平均支付金额,客单价类别
0,uid00324123,22.320,低客单价
1,uid00324124,87.000,高客单价
2,uid00324125,97.660,高客单价
3,uid00324126,33.365,低客单价
4,uid00324127,42.500,低客单价
...,...,...,...
25943,uid00350066,37.475,低客单价
25944,uid00350067,34.570,低客单价
25945,uid00350068,22.000,低客单价
25946,uid00350069,49.450,低客单价


In [13]:
# 把计算好的数据, 跟城市的数据拼接到一起
df_dup = df.drop_duplicates(subset=['用户ID','省份','城市'])
df_dup

,用户ID,付款日期,订单状态,实付金额,邮费,省份,城市,购买数量
0,uid00324123,2023-04-18,交易成功,22.320,0,北京,北京市,1
1,uid00324124,2023-02-17,交易成功,87.000,0,上海,上海市,1
2,uid00324125,2023-04-18,交易成功,97.660,0,福建省,福州市,2
3,uid00324126,2023-01-11,交易成功,37.230,0,河南省,安阳市,3
5,uid00324127,2023-06-16,交易成功,42.500,0,浙江省,衢州市,3
...,...,...,...,...,...,...,...,...
28827,uid00350066,2023-01-11,交易成功,37.475,0,上海,上海市,3
28828,uid00350067,2023-01-11,交易成功,34.570,0,山东省,烟台市,2
28829,uid00350068,2023-01-29,交易成功,22.000,0,江苏省,南京市,1
28830,uid00350069,2023-02-04,交易成功,49.450,0,上海,上海市,1


In [25]:
from pandas import DataFrame

df_merge:DataFrame = users_df.merge(df_dup,on='用户ID',how='left')[['用户ID','平均支付金额','客单价类别','省份','城市']]

In [18]:
df_merge

,用户ID,平均支付金额,客单价类别,省份,城市
0,uid00324123,22.320,低客单价,北京,北京市
1,uid00324124,87.000,高客单价,上海,上海市
2,uid00324125,97.660,高客单价,福建省,福州市
3,uid00324126,33.365,低客单价,河南省,安阳市
4,uid00324127,42.500,低客单价,浙江省,衢州市
...,...,...,...,...,...
26144,uid00350066,37.475,低客单价,上海,上海市
26145,uid00350067,34.570,低客单价,山东省,烟台市
26146,uid00350068,22.000,低客单价,江苏省,南京市
26147,uid00350069,49.450,低客单价,上海,上海市


In [27]:
# 统计每个城市高客单价, 低客单价的人数
# 透视表 margins参数 会做汇总, 把每一行的数据加到一起放在最右边(多一列), 每一列数据加到一起放到最下面(多一行)
result_df = df_merge.pivot_table(index = ['省份','城市'],columns = '客单价类别',values='用户ID',aggfunc='count',margins=True).reset_index()

In [29]:
# 计算每个城市高客单价的占比, 计算整体的高客单价占比   计算TGI
result_df['高客单价占比'] = result_df['高客单价']/result_df['All']

In [31]:
result_df.fillna(0,inplace=True)

In [33]:
result_df.iloc[result_df.shape[0]-1,result_df.shape[1]-1]

0.41557994569582013

In [34]:
result_df['总高客单价占比'] = result_df.iloc[result_df.shape[0]-1,result_df.shape[1]-1]

In [36]:
result_df['TGI'] = result_df['高客单价占比']/result_df['总高客单价占比'] * 100

In [37]:
result_df.sort_values(by='TGI',ascending=False)

客单价类别,省份,城市,低客单价,高客单价,All,高客单价占比,总高客单价占比,TGI
236,海南省,陵水黎族自治县,0.0,2.0,2,1.0,0.41558,240.627588
293,西藏自治区,昌都市,0.0,1.0,1,1.0,0.41558,240.627588
246,湖北省,神农架林区,0.0,1.0,1,1.0,0.41558,240.627588
235,海南省,琼海市,0.0,1.0,1,1.0,0.41558,240.627588
234,海南省,琼中黎族苗族自治县,0.0,1.0,1,1.0,0.41558,240.627588
...,...,...,...,...,...,...,...,...
228,海南省,儋州市,2.0,0.0,2,0.0,0.41558,0.000000
61,宁夏回族自治区,固原市,1.0,0.0,1,0.0,0.41558,0.000000
157,新疆维吾尔自治区,阿拉尔市,1.0,0.0,1,0.0,0.41558,0.000000
147,新疆维吾尔自治区,博尔塔拉蒙古自治州,2.0,0.0,2,0.0,0.41558,0.000000


In [40]:
# 根据TGI排序, 找到TGI比较高的城市, 这里需要过滤掉数据量太少的城市
result_df[result_df['All']>result_df['All'].mean()].sort_values(by='TGI',ascending=False)

客单价类别,省份,城市,低客单价,高客单价,All,高客单价占比,总高客单价占比,TGI
287,福建省,福州市,136.0,146.0,282,0.517730,0.41558,124.580241
27,北京,北京市,1301.0,1209.0,2510,0.481673,0.41558,115.903886
283,福建省,厦门市,119.0,105.0,224,0.468750,0.41558,112.794182
111,广东省,佛山市,135.0,119.0,254,0.468504,0.41558,112.734972
46,四川省,成都市,335.0,289.0,624,0.463141,0.41558,111.444508
0,上海,上海市,2823.0,2389.0,5212,0.458365,0.41558,110.295339
164,江苏省,无锡市,162.0,135.0,297,0.454545,0.41558,109.376176
120,广东省,深圳市,530.0,441.0,971,0.454171,0.41558,109.286062
112,广东省,广州市,658.0,537.0,1195,0.449372,0.41558,108.131393
216,浙江省,温州市,125.0,101.0,226,0.446903,0.41558,107.537108
